In [25]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [26]:
df = pd.read_csv("loan_data.csv")
df = df.dropna()

In [27]:
features = [
    "credit_lines_outstanding",
    "fico_score",
    "loan_amt_outstanding",
    "total_debt_outstanding",
    "income"
]

X = df[features]
y = df["default"]

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [22]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [29]:
log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train)

log_pd = log_model.predict_proba(X_test_scaled)[:, 1]

In [30]:
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train_scaled, y_train)

rf_pd = rf_model.predict_proba(X_test_scaled)[:, 1]

In [31]:
log_auc = roc_auc_score(y_test, log_pd)
rf_auc = roc_auc_score(y_test, rf_pd)

print("Model Comparison:")
print(f"Logistic Regression AUC: {log_auc:.4f}")
print(f"Random Forest AUC: {rf_auc:.4f}")

Model Comparison:
Logistic Regression AUC: 0.9988
Random Forest AUC: 0.9978


In [32]:
# Logistic Regression chosen because:
# - More interpretable for credit risk
# - Stable probability estimates
# - Preferred in regulatory environments

best_model = log_model

In [33]:
def expected_loss(model, borrower_df, scaler, recovery_rate=0.10):

    LGD = 1 - recovery_rate  # Loss Given Default

    borrower_df = borrower_df[features]  # enforce correct order

    X = scaler.transform(borrower_df)

    pd = model.predict_proba(X)[:, 1][0]

    ead = borrower_df["loan_amt_outstanding"].values[0]

    el = pd * LGD * ead

    return {
        "PD": pd,
        "Expected_Loss": el
    }

In [34]:
sample_borrower = pd.DataFrame([{
    "credit_lines_outstanding": 2,
    "fico_score": 700,
    "loan_amt_outstanding": 10000,
    "total_debt_outstanding": 20000,
    "income": 50000
}])

result = expected_loss(best_model, sample_borrower, scaler)

print(result)

{'PD': np.float64(0.9998195606866721), 'Expected_Loss': np.float64(8998.376046180048)}


In [35]:
{
    "PD": 0.12,
    "Expected_Loss": 1080.0
}

{'PD': 0.12, 'Expected_Loss': 1080.0}